Data Loading Notebook
===

Prepare data for the analysis. The raw data is downloaded from the FAIR Universe HiggsML challenge repository. Use the HiggsML package to download and process the dataset, followed by selections and saving to local cache.

This notebook is only to be run once to get the FAIR Universe dataset into `.root` ntuples. The rest of the workflow is independent of this.

#### Note: you can choose not to run the data processing steps if you preform a `git lfs pull` to download the pre-saved and pre-processed datasets.

In [10]:
import os, sys
sys.path.append("")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mplhep as hep
import yaml
import uproot

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import HiggsML
from HiggsML.systematics import systematics
from HiggsML.datasets import Data

hep.style.use(hep.style.ATLAS)

from HiggsML.datasets import download_dataset

In [11]:
def load_config(path: str) -> dict:
    with open(path, "r") as f:
        return yaml.safe_load(f)

In [12]:
data_config = load_config("config.pipeline.yaml")["data_loader"]

In [13]:
# data = download_dataset("https://zenodo.org/records/15131565/files/FAIR_Universe_HiggsML_data.zip")

data_input_dir = data_config['data']['data_input_dir']
data         = Data(data_input_dir)

2026-02-18 23:05:39,055 - HiggsML.datasets     - INFO     - Total rows: 220099101
2026-02-18 23:05:39,056 - HiggsML.datasets     - INFO     - Test size: 66029730


In [14]:
list_of_processes = data_config["data"]["processes"]
print(f"List of production modes to keep: {list_of_processes}")

List of production modes to keep: ['htautau', 'ztautau', 'ttbar']


In [15]:
selections = data_config["preprocess"]["selections"]
print(f"Event selections: {selections}")

Event selections: DER_mass_transverse_met_lep <= 250.0 and DER_mass_vis <= 500.0 and DER_sum_pt <= 1000 and DER_pt_tot <= 250 and DER_deltar_had_lep <= 4.5 and DER_pt_h <= 400 and DER_pt_ratio_lep_had <= 9.0



In [16]:
seed = data_config["preprocess"]["seed"]

In [17]:
train_fraction = data_config["data"]["train_size"]

data.load_train_set(train_size=train_fraction)
df_training_full = data.get_train_set()

del data

2026-02-18 23:05:53,148 - HiggsML.datasets     - INFO     - Selected train size: 53924279
2026-02-18 23:11:54,380 - HiggsML.datasets     - INFO     - Data loaded successfully


In [18]:
print(f"Class balance before processing: \n\n {df_training_full["detailed_labels"].value_counts()}")

Class balance before processing: 

 detailed_labels
ztautau    34427794
htautau    17848744
ttbar       1517526
diboson      130215
Name: count, dtype: int64


In [19]:
def process_data(df: pd.DataFrame, list_of_processes: list, seed: int) -> pd.DataFrame:
    """
    Filter to the requested processes and balance by downsampling to the smallest class.
    """

    all_labels = df["detailed_labels"].unique()
    processes_to_exclude = list(set(all_labels) - set(list_of_processes))
    print(f"Excluding processes: {processes_to_exclude}")

    df_filtered = df[~np.isin(df["detailed_labels"], processes_to_exclude)].copy()

    counts = df_filtered["detailed_labels"].value_counts()
    print(f"Counts before balancing:\n{counts}")

    # Downsample every process to the size of the smallest one,
    # rescaling weights so the total weight sum is preserved per process.
    min_process = counts.idxmin()
    n_min = counts.min()
    print(f"Balancing to minimum process count ('{min_process}'): {n_min}")

    df_list = []
    for _, df_process in df_filtered.groupby("detailed_labels"):
        weight_sum_orig = df_process["weights"].sum() 
        df_sampled = df_process.sample(n=n_min, random_state=seed)
        df_sampled = df_sampled.copy()                  
        df_sampled["weights"] *= weight_sum_orig / df_sampled["weights"].sum()
        df_list.append(df_sampled)

    return pd.concat(df_list).reset_index(drop=True)


In [20]:
# Find the "balanced" dataset
df_balanced = process_data(df_training_full, list_of_processes, seed)
print(f"Class balance after processing: \n\n {df_balanced["detailed_labels"].value_counts()}")

del df_training_full

Excluding processes: ['diboson']
Counts before balancing:
detailed_labels
ztautau    34427794
htautau    17848744
ttbar       1517526
Name: count, dtype: int64
Balancing to minimum process count ('ttbar'): 1517526
Class balance after processing: 

 detailed_labels
htautau    1517526
ttbar      1517526
ztautau    1517526
Name: count, dtype: int64


In [21]:
def apply_systematics(df: pd.DataFrame, syst_settings: dict) -> dict:
    """Generate the nominal dataset and all systematic variations."""
    dataset_dict = {}

    print("Generating nominal dataset...")
    dataset_dict["nominal"] = systematics(data_set=df, dopostprocess=False)

    for sample_name, syst_args in syst_settings.items():
        print(f"Generating systematic variation: {sample_name}")
        dataset_dict[sample_name] = systematics(data_set=df, dopostprocess=False, **syst_args)

    return dataset_dict

In [22]:
# Inject the shared seed into each systematic's data_config
syst_settings = data_config["systematics"]
for syst_cfg in syst_settings.values():
    syst_cfg["seed"] = seed

# Apply systematic variations
dataset_dict = apply_systematics(df_balanced, syst_settings)

Generating nominal dataset...
Generating systematic variation: TES_up
Generating systematic variation: TES_dn
Generating systematic variation: JES_up
Generating systematic variation: JES_dn


In [23]:
def save_root_files(
    dataset_dict: dict,
    output_dir: str,
    processes: list,
    selections: str,
) -> None:
    """
    Save each (sample, process) combination to its own ROOT file.
    """
    os.makedirs(output_dir, exist_ok=True)  
    print(f"Output directory: {output_dir}")

    for sample, df in dataset_dict.items():
        output_path = os.path.join(output_dir, f"dataset_{sample}.root")
        print(f"Writing {output_path}...")

        with uproot.recreate(output_path) as root_file:

            for process in processes:
                df_process = df[df["detailed_labels"] == process].copy()
                df_process = df_process.query(selections)

                # Drop the label column — not needed in the output tree
                columns_to_keep = [c for c in df_process.columns if c != "detailed_labels"]
                arrays = {col: df_process[col].to_numpy() for col in columns_to_keep}

                if arrays and len(df_process) > 0:
                    tree_name = f"tree_{process}"      
                    root_file[tree_name] = arrays
                    print(f"  Wrote tree '{tree_name}' with {len(df_process)} events.")
                else:
                    logger.warning(
                        f"  No events for process '{process}' in sample '{sample}' "
                        f"after selection — tree not written."
                    )

In [24]:
save_root_files(
            dataset_dict,
            data_config["output"]["dir"],
            list_of_processes,
            selections,
        )

Output directory: ./saved_datasets
Writing ./saved_datasets/dataset_nominal.root...
  Wrote tree 'tree_htautau' with 1509107 events.
  Wrote tree 'tree_ztautau' with 1517050 events.
  Wrote tree 'tree_ttbar' with 1476394 events.
Writing ./saved_datasets/dataset_TES_up.root...
  Wrote tree 'tree_htautau' with 1509108 events.
  Wrote tree 'tree_ztautau' with 1517050 events.
  Wrote tree 'tree_ttbar' with 1476403 events.
Writing ./saved_datasets/dataset_TES_dn.root...
  Wrote tree 'tree_htautau' with 1509113 events.
  Wrote tree 'tree_ztautau' with 1517050 events.
  Wrote tree 'tree_ttbar' with 1476387 events.
Writing ./saved_datasets/dataset_JES_up.root...
  Wrote tree 'tree_htautau' with 1509047 events.
  Wrote tree 'tree_ztautau' with 1517048 events.
  Wrote tree 'tree_ttbar' with 1476317 events.
Writing ./saved_datasets/dataset_JES_dn.root...
  Wrote tree 'tree_htautau' with 1509193 events.
  Wrote tree 'tree_ztautau' with 1517050 events.
  Wrote tree 'tree_ttbar' with 1476474 events.